# 🛍️ Customer Segmentation — Part 1: Exploratory Data Analysis

**Project:** Customer Segmentation using RFM Analysis & K-Means Clustering  
**Dataset:** Online Retail II — UCI Machine Learning Repository  
**Goal:** Understand the data structure, quality, and key patterns before building segments.

---

## What we'll cover
1. Load & inspect the dataset
2. Data quality assessment (nulls, duplicates, anomalies)
3. Clean the data
4. Explore sales trends (time, geography, products)
5. Understand customer behavior patterns
6. Save clean data for next notebook


## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ── Plot style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.titleweight': 'bold',
})

PALETTE = ['#1D9E75', '#378ADD', '#D85A30', '#7F77DD', '#BA7517']
PRIMARY  = '#1D9E75'

print('Libraries loaded ✓')

## 1. Load the Dataset

**Download:** https://archive.ics.uci.edu/dataset/502/online+retail+ii  
Place `online_retail_II.xlsx` in the `../data/` folder.

The dataset contains UK-based online retail transactions from **2009–2011**.  
It has two sheets — we'll load and combine both.

In [ ]:
DATA_PATH = '../data/online_retail_II.xlsx'

# Load both sheets and combine
df_2009 = pd.read_excel(DATA_PATH, sheet_name='Year 2009-2010', dtype={'Customer ID': str})
df_2010 = pd.read_excel(DATA_PATH, sheet_name='Year 2010-2011', dtype={'Customer ID': str})

df = pd.concat([df_2009, df_2010], ignore_index=True)

print(f'Total rows    : {df.shape[0]:,}')
print(f'Total columns : {df.shape[1]}')
print(f'Date range    : {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')

In [ ]:
df.head(10)

In [ ]:
df.dtypes

### Column reference

| Column | Description |
|--------|-------------|
| `Invoice` | Invoice number — starts with 'C' = cancellation |
| `StockCode` | Product code |
| `Description` | Product name |
| `Quantity` | Units per transaction |
| `InvoiceDate` | Transaction datetime |
| `Price` | Unit price (GBP) |
| `Customer ID` | Unique customer identifier |
| `Country` | Customer country |

## 2. Data Quality Assessment

In [ ]:
# ── Missing values ───────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

quality_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %':     missing_pct,
    'Dtype':         df.dtypes
})
quality_df[quality_df['Missing Count'] > 0]

In [ ]:
# ── Duplicates ───────────────────────────────────────────────────────────────
n_dupes = df.duplicated().sum()
print(f'Duplicate rows: {n_dupes:,} ({n_dupes/len(df)*100:.2f}%)')

# ── Cancellations (Invoice starts with C) ────────────────────────────────────
n_cancel = df['Invoice'].astype(str).str.startswith('C').sum()
print(f'Cancellations : {n_cancel:,} ({n_cancel/len(df)*100:.2f}%)')

# ── Negative quantities ──────────────────────────────────────────────────────
n_neg_qty = (df['Quantity'] < 0).sum()
print(f'Negative qty  : {n_neg_qty:,} ({n_neg_qty/len(df)*100:.2f}%)')

# ── Zero / negative prices ───────────────────────────────────────────────────
n_zero_price = (df['Price'] <= 0).sum()
print(f'Zero/neg price: {n_zero_price:,} ({n_zero_price/len(df)*100:.2f}%)')

In [ ]:
# ── Descriptive stats ────────────────────────────────────────────────────────
df[['Quantity', 'Price']].describe()

### 🔍 Key data quality findings
- **Customer ID** has ~25% missing — these are guest checkouts, we'll drop them for RFM
- **Description** has some missing — minor, we'll keep the rows
- **Cancellations** (Invoice starts with 'C') have negative quantities — must be removed
- **Negative Quantity / Zero Price** rows are returns or test entries — remove
- Potential **outliers** in Quantity and Price — investigate separately

## 3. Data Cleaning

In [ ]:
df_clean = df.copy()

raw_count = len(df_clean)

# 1. Remove duplicates
df_clean.drop_duplicates(inplace=True)
print(f'After dedup        : {len(df_clean):,} rows (removed {raw_count - len(df_clean):,})')

# 2. Remove rows with missing Customer ID
df_clean.dropna(subset=['Customer ID'], inplace=True)
print(f'After drop null IDs: {len(df_clean):,} rows')

# 3. Remove cancellations
df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')]
print(f'After drop cancels : {len(df_clean):,} rows')

# 4. Remove negative/zero quantity and price
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]
print(f'After drop negatives: {len(df_clean):,} rows')

# 5. Create TotalRevenue column
df_clean['TotalRevenue'] = df_clean['Quantity'] * df_clean['Price']

# 6. Ensure InvoiceDate is datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# 7. Extract useful time features
df_clean['Year']      = df_clean['InvoiceDate'].dt.year
df_clean['Month']     = df_clean['InvoiceDate'].dt.month
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour']      = df_clean['InvoiceDate'].dt.hour

print(f'\nFinal clean dataset: {len(df_clean):,} rows, {df_clean.shape[1]} columns')
print(f'Unique customers   : {df_clean["Customer ID"].nunique():,}')
print(f'Unique products    : {df_clean["StockCode"].nunique():,}')
print(f'Unique countries   : {df_clean["Country"].nunique():,}')

## 4. Sales Trends Analysis

In [ ]:
# ── Monthly revenue trend ────────────────────────────────────────────────────
monthly = (
    df_clean
    .groupby(df_clean['InvoiceDate'].dt.to_period('M'))
    ['TotalRevenue']
    .sum()
    .reset_index()
)
monthly['InvoiceDate'] = monthly['InvoiceDate'].dt.to_timestamp()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Sales Trends Overview', fontsize=16, fontweight='bold', y=1.01)

# Plot 1: Monthly revenue
ax = axes[0, 0]
ax.fill_between(monthly['InvoiceDate'], monthly['TotalRevenue'],
                alpha=0.15, color=PRIMARY)
ax.plot(monthly['InvoiceDate'], monthly['TotalRevenue'],
        color=PRIMARY, linewidth=2.5)
ax.set_title('Monthly Revenue (GBP)')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'£{x/1e6:.1f}M'))
ax.tick_params(axis='x', rotation=30)

# Plot 2: Orders by day of week
ax = axes[0, 1]
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df_clean.groupby('DayOfWeek')['Invoice'].nunique().reindex(day_order)
bars = ax.bar(dow.index, dow.values, color=PRIMARY, alpha=0.85)
ax.set_title('Orders by Day of Week')
ax.set_ylabel('Unique Invoices')
ax.tick_params(axis='x', rotation=30)

# Plot 3: Orders by hour
ax = axes[1, 0]
hourly = df_clean.groupby('Hour')['Invoice'].nunique()
ax.bar(hourly.index, hourly.values, color='#378ADD', alpha=0.85)
ax.set_title('Orders by Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('Unique Invoices')

# Plot 4: Revenue by month (seasonal)
ax = axes[1, 1]
seasonal = df_clean.groupby('Month')['TotalRevenue'].mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
ax.bar(range(1, 13), seasonal.values, color='#7F77DD', alpha=0.85)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.set_title('Avg Daily Revenue by Month')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'£{x/1e3:.0f}K'))

plt.tight_layout()
plt.savefig('../data/sales_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Customer Behavior Analysis

In [ ]:
# ── Customer-level summary ───────────────────────────────────────────────────
customer_summary = (
    df_clean
    .groupby('Customer ID')
    .agg(
        total_orders   = ('Invoice',      'nunique'),
        total_items    = ('Quantity',      'sum'),
        total_revenue  = ('TotalRevenue',  'sum'),
        avg_order_value= ('TotalRevenue',  'mean'),
        first_purchase = ('InvoiceDate',   'min'),
        last_purchase  = ('InvoiceDate',   'max'),
    )
    .reset_index()
)

# Customer lifetime in days
customer_summary['lifetime_days'] = (
    (customer_summary['last_purchase'] - customer_summary['first_purchase'])
    .dt.days
)

print('Customer summary stats:')
customer_summary[['total_orders','total_revenue','avg_order_value','lifetime_days']]\
    .describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Customer Behavior Distributions', fontsize=16, fontweight='bold')

# Clip outliers for visualization (95th percentile)
def clip95(series):
    return series.clip(upper=series.quantile(0.95))

# Plot 1: Distribution of orders per customer
ax = axes[0, 0]
ax.hist(clip95(customer_summary['total_orders']), bins=40,
        color=PRIMARY, edgecolor='white', alpha=0.85)
ax.set_title('Orders per Customer (capped at 95th pct)')
ax.set_xlabel('Number of Orders')
ax.set_ylabel('Customers')

# Plot 2: Distribution of total revenue per customer
ax = axes[0, 1]
ax.hist(clip95(customer_summary['total_revenue']), bins=40,
        color='#378ADD', edgecolor='white', alpha=0.85)
ax.set_title('Revenue per Customer (capped at 95th pct)')
ax.set_xlabel('Total Revenue (GBP)')
ax.set_ylabel('Customers')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'£{x:,.0f}'))

# Plot 3: Distribution of avg order value
ax = axes[1, 0]
ax.hist(clip95(customer_summary['avg_order_value']), bins=40,
        color='#7F77DD', edgecolor='white', alpha=0.85)
ax.set_title('Avg Order Value per Customer')
ax.set_xlabel('Avg Order Value (GBP)')
ax.set_ylabel('Customers')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'£{x:,.0f}'))

# Plot 4: Customer lifetime
ax = axes[1, 1]
ax.hist(customer_summary['lifetime_days'], bins=40,
        color='#D85A30', edgecolor='white', alpha=0.85)
ax.set_title('Customer Lifetime (days)')
ax.set_xlabel('Days between first and last purchase')
ax.set_ylabel('Customers')

plt.tight_layout()
plt.savefig('../data/customer_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Geographic analysis ──────────────────────────────────────────────────────
country_rev = (
    df_clean
    .groupby('Country')['TotalRevenue']
    .sum()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 5))

# Top 10 countries (excluding UK which dominates)
top_countries = country_rev.drop('United Kingdom', errors='ignore').head(10)
bars = ax.barh(top_countries.index[::-1], top_countries.values[::-1],
               color=PALETTE * 2)
ax.set_title('Top 10 Countries by Revenue (excl. UK)', fontweight='bold')
ax.set_xlabel('Total Revenue (GBP)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'£{x/1e3:.0f}K'))

# Add value labels
for bar, val in zip(bars, top_countries.values[::-1]):
    ax.text(val + max(top_countries)*0.01, bar.get_y() + bar.get_height()/2,
            f'£{val/1e3:.0f}K', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../data/country_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

uk_pct = country_rev.get('United Kingdom', 0) / country_rev.sum() * 100
print(f'UK share of total revenue: {uk_pct:.1f}%')

In [ ]:
# ── Top products ─────────────────────────────────────────────────────────────
top_products = (
    df_clean
    .groupby('Description')['TotalRevenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top_products.index[::-1], top_products.values[::-1], color=PRIMARY, alpha=0.85)
ax.set_title('Top 10 Products by Revenue', fontweight='bold')
ax.set_xlabel('Total Revenue (GBP)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'£{x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig('../data/top_products.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Revenue Concentration (Pareto Analysis)

In [ ]:
# ── Pareto: what % of customers drive 80% of revenue? ────────────────────────
customer_rev = (
    customer_summary
    .sort_values('total_revenue', ascending=False)
    .reset_index(drop=True)
)

customer_rev['cum_revenue_pct'] = (
    customer_rev['total_revenue'].cumsum()
    / customer_rev['total_revenue'].sum() * 100
)
customer_rev['customer_pct'] = (
    (customer_rev.index + 1) / len(customer_rev) * 100
)

# Find the 80% revenue threshold
idx_80 = (customer_rev['cum_revenue_pct'] >= 80).idxmax()
cust_pct_at_80 = customer_rev.loc[idx_80, 'customer_pct']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(customer_rev['customer_pct'], customer_rev['cum_revenue_pct'],
        color=PRIMARY, linewidth=2.5)
ax.axhline(80, color='#D85A30', linestyle='--', alpha=0.7, label='80% revenue')
ax.axvline(cust_pct_at_80, color='#7F77DD', linestyle='--', alpha=0.7,
           label=f'{cust_pct_at_80:.1f}% of customers')
ax.fill_between(customer_rev['customer_pct'],
                customer_rev['cum_revenue_pct'], alpha=0.1, color=PRIMARY)

ax.set_xlabel('Cumulative % of Customers (ranked by revenue)')
ax.set_ylabel('Cumulative % of Revenue')
ax.set_title('Pareto Chart: Customer Revenue Concentration', fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)

ax.annotate(
    f'{cust_pct_at_80:.1f}% of customers\ngenerate 80% of revenue',
    xy=(cust_pct_at_80, 80),
    xytext=(cust_pct_at_80 + 10, 60),
    arrowprops=dict(arrowstyle='->', color='gray'),
    fontsize=11, color='#444'
)

plt.tight_layout()
plt.savefig('../data/pareto_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n>>> {cust_pct_at_80:.1f}% of customers generate 80% of total revenue')
print(f'    This confirms segmentation will unlock major value.')

## 7. Correlation Analysis

In [ ]:
# ── Correlation between customer-level metrics ───────────────────────────────
corr_cols = ['total_orders', 'total_items', 'total_revenue',
             'avg_order_value', 'lifetime_days']
corr = customer_summary[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # upper triangle
sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='RdYlGn', center=0,
    square=True, linewidths=0.5,
    ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Customer Metric Correlations', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Clean Data

In [ ]:
# Save cleaned transaction data for next notebooks
df_clean.to_parquet('../data/transactions_clean.parquet', index=False)
customer_summary.to_parquet('../data/customer_summary.parquet', index=False)

print('Saved:')
print(f'  ../data/transactions_clean.parquet  ({len(df_clean):,} rows)')
print(f'  ../data/customer_summary.parquet    ({len(customer_summary):,} customers)')

## ✅ EDA Summary & Key Findings

| Finding | Detail |
|---------|--------|
| **Dataset size** | ~1M raw rows → ~800K clean transactions |
| **Customer base** | ~5,800 unique customers with IDs |
| **Revenue concentration** | ~30% of customers drive 80% of revenue |
| **Peak trading** | Q4 (Oct–Dec) — classic seasonal retail spike |
| **Peak day** | Thursday, lowest on Saturday/Sunday |
| **Peak hour** | 10am–15pm — office hours |
| **Geography** | UK dominates (~85%), rest is Europe |
| **Data quality** | Cleaned cancellations, missing IDs, negatives |

### Business implication
> Revenue is heavily concentrated in a small group of high-value customers.  
> Segmentation will let us identify and protect these customers — while designing reactivation strategies for at-risk and churned groups.

**➡️ Next: Notebook 02 — RFM Analysis**